In [ ]:
# GATモデルの定義 (5層)
class GATModel(torch.nn.Module):  # 変更
    def __init__(self, hidden_channels=128):
        super(GATModel, self).__init__()
        self.conv1 = GATConv(4, hidden_channels, heads=4)  
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels * 2, heads=4)  
        self.conv3 = GATConv(hidden_channels * 8, hidden_channels * 4, heads=4)  
        self.conv4 = GATConv(hidden_channels * 16, hidden_channels, heads=4)  
        self.conv5 = GATConv(hidden_channels * 4, hidden_channels)  # 新しい5層目
        self.fc_defect = torch.nn.Linear(hidden_channels, 1)  # 回帰タスク用に出力1次元

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = F.relu(self.conv4(x, edge_index))
        x = F.relu(self.conv5(x, edge_index))  # 新しい5層目
        x = self.fc_defect(x)  
        return torch.sigmoid(x)  

# ハイパーパラメータ設定
hidden_channels = 128
learning_rate = 0.001
batch_size = 32
epochs = 2000
weight_decay = 5e-4
patience = 500  # Early Stopping用

In [1]:
import os
import time
import json
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import datetime
from torch.utils.tensorboard import SummaryWriter

# デバイスの設定
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Using device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# データフォルダ
standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Normalized1"
label_data_folder = "/home/nishioka/GNN/Defectlabel4x4/DefectLabels_4x4_test1"

# 座標データのロード（最大3654行）
coords = ["x", "y", "z"]
coords_data = {coord: np.load(f"/home/nishioka/GNN/BasicdataforGNN/{coord}_2layer_normalized.npy")[:3654] for coord in coords}

# エッジ情報を読み込む
edges = np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy")
edge_index = torch.tensor(edges.T, dtype=torch.long).to(device)

# データとラベルファイルを対応付け
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Normalized1_Defect4x4_ELNOD")]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith("DefectLabel_L")]

# 欠陥なしデータのペア
defect_free_pair = ("/home/nishioka/GNN/BasicdataforGNN/Normalized1_nodefect_ElNOD.npy",
                    "/home/nishioka/GNN/BasicdataforGNN/DefectLabel_nodefect.npy")

# 初期のペアリスト
train_pairs = []
val_pairs = []
test_pairs = []

# 欠陥なしデータを追加
for _ in range(8):
 train_pairs.append(defect_free_pair)
val_pairs.append(defect_free_pair)
test_pairs.append(defect_free_pair)

# ペアのデータをカウントして確認
def count_pair_occurrences(pairs, target_pair):
    return pairs.count(target_pair)

train_count = count_pair_occurrences(train_pairs, defect_free_pair)
val_count = count_pair_occurrences(val_pairs, defect_free_pair)
test_count = count_pair_occurrences(test_pairs, defect_free_pair)

print(f"\nDefect-free data occurrences in train_pairs: {train_count} (expected: 8)")
print(f"Defect-free data occurrences in val_pairs: {val_count} (expected: 1)")
print(f"Defect-free data occurrences in test_pairs: {test_count} (expected: 1)")

if train_count == 8 and val_count == 1 and test_count == 1:
    print("\nAll defect-free data pairs are correctly added.")
else:
    print("\nThere is an issue with adding defect-free data pairs.")

# データとラベルのペア作成
def extract_layer_block(file_name):
    """ファイル名から層とブロック番号を抽出"""
    if file_name.startswith("0"):
        return (0, 0)
    try:
        layer_block_str = file_name.split("_")[-1].replace(".npy", "")
        layer = int(layer_block_str.split("L")[1].split("B")[0])
        block = int(layer_block_str.split("B")[1])
        return (layer, block)
    except (ValueError, IndexError):
        print(f"Invalid file name format: {file_name}")
        return None

data_label_pairs = {}
for data_file in data_files:
    layer_block = extract_layer_block(data_file)
    if layer_block:
        data_label_pairs[layer_block] = {"data": data_file}

for label_file in label_files:
    layer_block = extract_layer_block(label_file)
    if layer_block and layer_block in data_label_pairs:
        data_label_pairs[layer_block]["label"] = label_file

# データとラベルのペアを作成
data_label_pairs = {}
for data_file in data_files:
    layer_block = extract_layer_block(data_file)
    if layer_block:
        data_label_pairs[layer_block] = {"data": data_file}

for label_file in label_files:
    layer_block = extract_layer_block(label_file)
    if layer_block and layer_block in data_label_pairs:
        data_label_pairs[layer_block]["label"] = label_file

# 有効なペアのみ取得
valid_pairs = [(v["data"], v["label"]) for k, v in data_label_pairs.items() if "label" in v]

# ランダムにサンプリング
num_samples = 1296
random_indices = np.random.choice(len(valid_pairs), num_samples, replace=False)

# サンプルをトレーニングデータとして取得
train_pairs = [valid_pairs[i] for i in random_indices]

# 残りのデータを取得
remaining_pairs = [valid_pairs[i] for i in range(len(valid_pairs)) if i not in random_indices]

# 残りのデータを1:1でバリデーションとテストに分割
val_pairs, test_pairs = train_test_split(remaining_pairs, test_size=0.5, random_state=42)

# 検証データとテストデータを適切に分けて格納
train_data = [pair[0] for pair in train_pairs]
train_labels = [pair[1] for pair in train_pairs]

val_data = [pair[0] for pair in val_pairs]
val_labels = [pair[1] for pair in val_pairs]

test_data = [pair[0] for pair in test_pairs]
test_labels = [pair[1] for pair in test_pairs]


# データ準備関数
def prepare_data(pairs):
    """データとラベルを準備し、テンソルに変換"""
    sampled_data, sampled_labels = [], []
    for data_file, label_file in pairs:
        data_path = os.path.join(standardized_data_folder, data_file)
        label_path = os.path.join(label_data_folder, label_file)

        values = np.load(data_path)[:3654]
        label = np.load(label_path)[:3654]

        node_features = np.vstack((coords_data["x"], coords_data["y"], coords_data["z"], values)).T
        sampled_data.append(node_features)
        sampled_labels.append(label)

    sampled_data = np.stack(sampled_data)
    sampled_labels = np.hstack(sampled_labels)

    x = torch.tensor(sampled_data.reshape(-1, 4), dtype=torch.float).to(device)
    y = torch.tensor(sampled_labels, dtype=torch.float).to(device)

    return x, y

# 各データセットを準備
train_x, train_y = prepare_data(train_pairs)
val_x, val_y = prepare_data(val_pairs)
test_x, test_y = prepare_data(test_pairs)

# Dataオブジェクト作成
train_data = Data(x=train_x, edge_index=edge_index, y=train_y)
val_data = Data(x=val_x, edge_index=edge_index, y=val_y)
test_data = Data(x=test_x, edge_index=edge_index, y=test_y)

# モデル定義
class GATModel(torch.nn.Module):
    def __init__(self, hidden_channels=128):
        super(GATModel, self).__init__()
        self.conv1 = GATConv(4, hidden_channels, heads=4)
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels * 2, heads=4)
        self.conv3 = GATConv(hidden_channels * 8, hidden_channels , heads=4)
        self.conv4 = GATConv(hidden_channels * 4, hidden_channels)
        # self.conv5 = GATConv(hidden_channels * 4, hidden_channels)
        self.fc_defect = torch.nn.Linear(hidden_channels, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = F.relu(self.conv4(x, edge_index))
        # x = F.relu(self.conv5(x, edge_index))
        x = self.fc_defect(x)
        return torch.sigmoid(x)

# モデル、損失関数、オプティマイザの設定
model = GATModel(hidden_channels=128).to(device)

# DPを使用する場合、モデルをラップする
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with Data Parallel")
    model = torch.nn.DataParallel(model)

# モデルの初期化
model.apply(lambda m: torch.nn.init.xavier_uniform_(m.weight) if isinstance(m, torch.nn.Linear) else None)

# オプティマイザと損失関数の設定
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
loss_fn = torch.nn.MSELoss()


# TensorBoard設定
writer = SummaryWriter(log_dir=f'/home/nishioka/GNN/GNNlogs/{type(model).__name__}_{timestamp}')

# データローダーの作成
train_loader = DataLoader([train_data], batch_size=16, shuffle=True)
val_loader = DataLoader([val_data], batch_size=16)
test_loader = DataLoader([test_data], batch_size=16)

# Early Stopping定義
class EarlyStopping:
    def __init__(self, patience=100, path='/home/nishioka/GNN/{type(model).__name__}_checkpoint.pt'):
        self.patience = patience
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.path = path
        self.counter = 0

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None or score > self.best_score:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

# 学習
early_stopping = EarlyStopping()
train_losses, val_losses = [], []

for epoch in range(1, 2001):
    model.train()
    train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch.y.view(-1, 1))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_losses.append(train_loss / len(train_loader))

    # 検証
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch)
            loss = loss_fn(out, batch.y.view(-1, 1))
            val_loss += loss.item()
    val_losses.append(val_loss / len(val_loader))

    early_stopping(val_loss / len(val_loader), model)
    if early_stopping.early_stop:
        print(f"Early stopping at epoch {epoch}")
        break

    if epoch % 50 == 0:
        print(f'Epoch {epoch}, Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}')

# テストデータで評価
model.load_state_dict(torch.load(early_stopping.path))
model.eval()
test_loss, test_preds, test_targets = 0, [], []
with torch.no_grad():
    for batch in test_loader:
        out = model(batch)
        loss = loss_fn(out, batch.y.view(-1, 1))
        test_loss += loss.item()
        test_preds.append(out.cpu().numpy())
        test_targets.append(batch.y.cpu().numpy())

# 評価指標の計算
test_preds, test_targets = np.concatenate(test_preds), np.concatenate(test_targets)
rmse = np.sqrt(mean_squared_error(test_targets, test_preds))
mae = mean_absolute_error(test_targets, test_preds)
r2 = r2_score(test_targets, test_preds)

print(f'Test Loss: {test_loss:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2 Score: {r2:.4f}')

# 結果の保存やプロットは省略

# Training Summary
print(f"Training completed in {(time.time() - start_time):.2f} seconds.")


# データの保存
np.save(f'/home/nishioka/GNN/4x4trainindata_confirm1/train_pairs{type(model).__name__}_{timestamp}.npy', train_pairs)
np.save(f'/home/nishioka/GNN/4x4trainindata_confirm1/val_pairs{type(model).__name__}_{timestamp}.npy', val_pairs)
np.save(f'/home/nishioka/GNN/4x4trainindata_confirm1/test_pairs{type(model).__name__}_{timestamp}.npy', test_pairs)

Using device: NVIDIA GeForce RTX 3090

Defect-free data occurrences in train_pairs: 8 (expected: 8)
Defect-free data occurrences in val_pairs: 1 (expected: 1)
Defect-free data occurrences in test_pairs: 1 (expected: 1)

All defect-free data pairs are correctly added.
Using 3 GPUs with Data Parallel


OutOfMemoryError: Caught OutOfMemoryError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/home/nishioka/miniconda3/envs/interstage_gnn_first/lib/python3.12/site-packages/torch/nn/parallel/parallel_apply.py", line 83, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nishioka/miniconda3/envs/interstage_gnn_first/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1511, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nishioka/miniconda3/envs/interstage_gnn_first/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1520, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2700854/230825470.py", line 181, in forward
    x = F.relu(self.conv1(x, edge_index))
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nishioka/miniconda3/envs/interstage_g